In [1]:
from src.connection import client, local_client
from src.utils import fetch_one_day_data, fetch_data

import json
import os
import datetime
import calendar

import time
import memory_profiler

%load_ext memory_profiler

# Fetch 20 items of each day from 2019-01-01 to 2021-12-31

### Fetch data from remote opensearch to local machine

In [2]:
index_name = 'frameintell_aux_minilmv6'

In [3]:
%%time
%%memit

for year in [2019, 2020, 2021]:
    for month in [1, 4, 7, 10]:
        first_day = f"{year}-{month}-01"
        _, last_day = calendar.monthrange(year, month + 2)
        last_day = f"{year}-{month+2}-{last_day}"

        filename = f"original_data_pubmed_{first_day}_{last_day}.json"
        if os.path.exists(os.path.join("data/original_data", filename)):
            print(f"{filename} exists.")
            continue

        print(f"==》fetch data from {first_day} to {last_day}")
        data = fetch_data(first_day, last_day, client,index_name)

        with open(os.path.join("data/original_data", filename), "w") as f:
            f.write(json.dumps(data))

==》fetch data from 2019-1-01 to 2019-3-31
==》fetch data from 2019-4-01 to 2019-6-30
==》fetch data from 2019-7-01 to 2019-9-30
==》fetch data from 2019-10-01 to 2019-12-31
==》fetch data from 2020-1-01 to 2020-3-31
==》fetch data from 2020-4-01 to 2020-6-30
==》fetch data from 2020-7-01 to 2020-9-30
==》fetch data from 2020-10-01 to 2020-12-31
==》fetch data from 2021-1-01 to 2021-3-31
==》fetch data from 2021-4-01 to 2021-6-30
==》fetch data from 2021-7-01 to 2021-9-30
==》fetch data from 2021-10-01 to 2021-12-31
peak memory: 150.54 MiB, increment: 92.23 MiB
CPU times: user 18.1 s, sys: 1.69 s, total: 19.8 s
Wall time: 1min 2s


### Bulk data to local opensearch

In [4]:
try:
    response = local_client.indices.create(index_name)
    print("Creating index:")
    print(response)
except Exception as e:
    print(e)

Creating index:
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'frameintell_aux_minilmv6'}


In [5]:
for year in [2019, 2020, 2021]:
    for month in [1, 4, 7, 10]:
        first_day = f"{year}-{month}-01"
        _, last_day = calendar.monthrange(year, month + 2)
        last_day = f"{year}-{month+2}-{last_day}"

        filename = f"original_data_pubmed_{first_day}_{last_day}.json"
        with open(os.path.join("data/original_data", filename), "r") as f:
            data = json.load(f)

        print(f"bulk data: {filename}")
        docs = []
        for item in data:
            item_index = {"index": {"_index": index_name, "_id": item["_id"]}}
            item_data = item['_source']
            
            docs.append(item_index)
            docs.append(item_data)

        resp = local_client.bulk(body=docs, index=index_name)

bulk data: original_data_pubmed_2019-1-01_2019-3-31.json
bulk data: original_data_pubmed_2019-4-01_2019-6-30.json
bulk data: original_data_pubmed_2019-7-01_2019-9-30.json
bulk data: original_data_pubmed_2019-10-01_2019-12-31.json
bulk data: original_data_pubmed_2020-1-01_2020-3-31.json
bulk data: original_data_pubmed_2020-4-01_2020-6-30.json
bulk data: original_data_pubmed_2020-7-01_2020-9-30.json
bulk data: original_data_pubmed_2020-10-01_2020-12-31.json
bulk data: original_data_pubmed_2021-1-01_2021-3-31.json
bulk data: original_data_pubmed_2021-4-01_2021-6-30.json
bulk data: original_data_pubmed_2021-7-01_2021-9-30.json
bulk data: original_data_pubmed_2021-10-01_2021-12-31.json


In [6]:
client.close()
local_client.close()